# 01. Web Scraping & Structural Text Normalization

This notebook provides a visual sanity check for the raw ingestion stage of the Vachanamrut RAG pipeline (`src/step1_scraper.py`).

### Key Demonstration Steps:
1. **Diacritic & Script Sanitization**: Stripping Devnagari/Gujarati non-ASCII artifacts while preserving English transliterated terms (e.g., *Vairagya*, *Atma*).
2. **DOM Parsing & Category Tagging**: Extracting web elements (`p`, `blockquote`, `li`) into semantically tagged blocks (`HEADER`, `SHLOKA`, `BODY`, `FOOTNOTE`).
3. **Structured Chapter Output**: Inspecting the raw json format generated before structural chunking.


In [1]:
import json
import sys
from pathlib import Path

# Add project root to path for modular imports
sys.path.append("..")

from src.step1_scraper import clean_and_strip_devnagari, fetch_vachanamrut

## Step 1: Text Sanitization Sanity Check

Test the `clean_and_strip_devnagari` function against text containing broken diacritics, accented Latin characters, and native Devnagari script to verify clean normalization.

In [2]:
# Sample noisy input containing diacritics and Devnagari text
raw_sample_text = "Gadhada I-1: Continuous āñḍ śānt Ātmā-realization (गadhada 1)"

cleaned_text = clean_and_strip_devnagari(raw_sample_text)

print("--- Text Sanitization Verification ---")
print(f"Raw Input : {raw_sample_text}")
print(f"Cleaned   : {cleaned_text}")

--- Text Sanitization Verification ---
Raw Input : Gadhada I-1: Continuous āñḍ śānt Ātmā-realization (गadhada 1)
Cleaned   : Gadhada I-1: Continuous and shant Atma-realization (adhada 1)


## Step 2: Live Fetch & Structural Block Parsing

Fetch a lightweight sample chapter (**Chapter 1: Gadhada I-1**) live from the website and inspect how HTML tags are converted into semantically prefixed blocks.

In [3]:
# Fetch Chapter 1 (Gadhada I-1)
sample_chapter_id = 1
parsed_blocks = fetch_vachanamrut(sample_chapter_id)

print(f"Total blocks extracted for Chapter {sample_chapter_id}: {len(parsed_blocks)}\n")
print("--- Sample Parsed Blocks (First 5) ---")
for block in parsed_blocks[:5]:
    prefix, text = block.split("::", 1)
    print(f"[{prefix:<8}] {text[:90]}...")

Total blocks extracted for Chapter 1: 18

--- Sample Parsed Blocks (First 5) ---
[HEADER  ] Gadhada I-1: Continuously Engaging One's Mind on God...
[BODY    ] On the night of Magshar sudi 4, Samvat 1876 [21 November 1819], Shriji Maharaj had come to...
[BODY    ] Thereupon Shriji Maharaj asked, “Which is the most difficult of all spiritual endeavors?”...
[BODY    ] The brahmacharis, sadhus and householders answered according to their own understanding, b...
[BODY    ] Shriji Maharaj then said, “Allow Me to answer. There is no spiritual endeavor more difficu...


## Step 3: Inspect Raw Output Schema

Inspect the final JSON structure for a chapter to verify metadata association (`chapter_id`, `title`, and `blocks`).

In [5]:
# Assemble sample chapter structure
sample_chapter_json = {
    "chapter_id": sample_chapter_id,
    "title": "Gadhada I-1: Continuously Engaging One's Mind on God",
    "blocks": parsed_blocks[:3],  # Showing first 3 blocks for brief display
}

print(json.dumps(sample_chapter_json, indent=2, ensure_ascii=False))

{
  "chapter_id": 1,
  "title": "Gadhada I-1: Continuously Engaging One's Mind on God",
  "blocks": [
    "HEADER::Gadhada I-1: Continuously Engaging One's Mind on God",
    "BODY::On the night of Magshar sudi 4, Samvat 1876 [21 November 1819], Shriji Maharaj had come to the residential hall of the sadhus in Dada Khachar’s darbar in Gadhada. He was dressed entirely in white clothes. At that time, an assembly of sadhus as well as devotees from various places had gathered before Him.",
    "BODY::Thereupon Shriji Maharaj asked, “Which is the most difficult of all spiritual endeavors?”"
  ]
}
